# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

**1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?

In [3]:
# value_counts() counts how many transactions are legit (0) and how many are fraud (1)
# normalize=True gives the same thing as a percentage
print(fraud["fraud"].value_counts())
print()
print(fraud["fraud"].value_counts(normalize=True))

# Answer:
# About 91% of the transactions are legit (0) and only about 9% are fraud (1).
# Yes, this is an IMBALANCED dataset: one class is much bigger than the other.
# Problem: a model could say "legit" every time and still get 91% accuracy,
# while catching ZERO frauds. So accuracy alone will not tell us if the model is good.

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

fraud
0.0    0.912597
1.0    0.087403
Name: proportion, dtype: float64


**2.** Train a LogisticRegression.

In [4]:
# X = the information about the transaction, y = fraud or not
X = fraud.drop(columns=["fraud"])
y = fraud["fraud"]

# 80% to train, 20% to test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling: put every column between 0 and 1 (fit on TRAIN only to avoid data leakage)
from sklearn.preprocessing import MinMaxScaler

normalizer = MinMaxScaler()
normalizer.fit(X_train)
X_train_norm = normalizer.transform(X_train)
X_test_norm = normalizer.transform(X_test)

# Logistic Regression = a classification model that gives the probability of fraud
# if the probability is above 50% -> it predicts fraud (1), otherwise legit (0)
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X_train_norm, y_train)
pred = log_reg.predict(X_test_norm)

**3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.

In [5]:
# classification_report shows precision, recall and f1-score for each class
# - precision (fraud): when the model says "fraud", how often is it really a fraud?
# - recall (fraud):    out of all the real frauds, how many did the model catch?
# - f1-score:          a mix of precision and recall
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

# Comment:
# - Accuracy looks great (0.95), but it is misleading because 91% of the data is legit.
# - The important class here is FRAUD (1): missing a fraud costs the bank money.
# - So the correct metric to look at is the RECALL of class 1 (how many frauds we catch).
# - Recall for fraud is only 0.41: the model misses almost 6 frauds out of 10!
# - Precision for fraud is high (0.91): when it says fraud, it's usually right,
#   but it is too "shy" and rarely says fraud, because it mostly saw legit examples.

              precision    recall  f1-score   support

         0.0       0.95      1.00      0.97    182557
         1.0       0.91      0.41      0.57     17443

    accuracy                           0.95    200000
   macro avg       0.93      0.71      0.77    200000
weighted avg       0.94      0.95      0.94    200000



**4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model?

In [6]:
# Oversampling = we copy random fraud rows until there are as many frauds as legit transactions
# IMPORTANT: we only resample the TRAIN data. The test data stays real (imbalanced),
# because in real life the model will see real, imbalanced data
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train_norm, y_train)

# Step 1 again: the target is now balanced 50/50
print(y_train_over.value_counts())
print()

# Step 2 again: train the same model on the balanced data
log_reg_over = LogisticRegression()
log_reg_over.fit(X_train_over, y_train_over)
pred_over = log_reg_over.predict(X_test_norm)

# Step 3 again: evaluate
print(classification_report(y_test, pred_over))

# Comment:
# - Yes, it improves a lot: recall for fraud goes from 0.41 to 0.93.
#   The model now catches about 93 frauds out of 100.
# - The f1-score for fraud also goes up (0.57 -> 0.69), so the model is better overall.
# - Price to pay: precision for fraud drops (0.91 -> 0.55), so there are more false alarms
#   (legit transactions flagged as fraud). For fraud detection, this trade-off is usually OK:
#   it's better to check a few extra transactions than to miss real frauds.

fraud
0.0    730040
1.0    730040
Name: count, dtype: int64

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182557
         1.0       0.55      0.93      0.69     17443

    accuracy                           0.93    200000
   macro avg       0.77      0.93      0.82    200000
weighted avg       0.95      0.93      0.93    200000



**5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?

In [7]:
# Undersampling = we REMOVE random legit rows until there are as many legit as frauds
# (again, only on the TRAIN data)
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train_norm, y_train)

# Step 1 again: balanced target (but much less data than before)
print(y_train_under.value_counts())
print()

# Step 2 again: train
log_reg_under = LogisticRegression()
log_reg_under.fit(X_train_under, y_train_under)
pred_under = log_reg_under.predict(X_test_norm)

# Step 3 again: evaluate
print(classification_report(y_test, pred_under))

# Comment:
# - Yes, it also improves the fraud recall a lot (0.41 -> 0.92).
# - But it is a bit worse than oversampling: precision for fraud is lower (0.47 vs 0.55),
#   so f1-score is 0.62 vs 0.69 -> more false alarms.
# - Why? We threw away about 90% of the legit rows (730,040 -> 69,960),
#   so the model learned less about what a normal transaction looks like.
# - Advantage: training is much faster (less data).

fraud
0.0    69960
1.0    69960
Name: count, dtype: int64

              precision    recall  f1-score   support

         0.0       0.99      0.90      0.94    182557
         1.0       0.47      0.92      0.62     17443

    accuracy                           0.90    200000
   macro avg       0.73      0.91      0.78    200000
weighted avg       0.95      0.90      0.91    200000



**6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?

In [8]:
# SMOTE = instead of COPYING frauds, it CREATES new, artificial frauds
# How: it takes a fraud, looks at its closest fraud neighbours (like KNN),
#      and creates a new point somewhere between them
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_norm, y_train)

# Step 1 again: balanced target
print(y_train_smote.value_counts())
print()

# Step 2 again: train
log_reg_smote = LogisticRegression()
log_reg_smote.fit(X_train_smote, y_train_smote)
pred_smote = log_reg_smote.predict(X_test_norm)

# Step 3 again: evaluate
print(classification_report(y_test, pred_smote))

# Comment:
# - Yes, SMOTE improves fraud recall too (0.41 -> 0.93), with the same results as oversampling
#   (precision 0.55, f1-score 0.69).
#
# Final conclusion:
# - Without balancing, the model only caught 41% of frauds.
# - All 3 balancing methods raise fraud recall to about 92-93%, with a lower precision (more false alarms).
# - Oversampling and SMOTE are the best here (f1-score 0.69), undersampling is slightly worse (0.62)
#   because it throws away a lot of useful data.
# - SMOTE gives the same result as oversampling because we have a lot of fraud examples already.
#   SMOTE is the most useful when there are very few examples of the small class.

fraud
0.0    730040
1.0    730040
Name: count, dtype: int64

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182557
         1.0       0.55      0.93      0.69     17443

    accuracy                           0.93    200000
   macro avg       0.77      0.93      0.82    200000
weighted avg       0.95      0.93      0.93    200000

